# Task 3 — Notebook 02: Baseline climatology + multi-year event anomalies

**Baseline:** ISO calendar weeks, years **2015–2021**, μ and σ per (`iy`, `ix`, `iso_week`).

**Events:** Configured in `configs/task3_soil_moisture.yaml` → **`event_windows`** (default: **2019** Midwest/MRV wet season; **2022** Great Plains Jun–Aug flash drought). Each window lists `year`, `start_date`, `end_date`, and optional **`duration_mode`** (`wet_above` vs `dry_below`) for the persistence map in notebook 03.

**z-score** = (obs − μ) / (σ + floor).

**Outputs:** `data/processed/task3/smap_climatology.parquet` (once), then **`smap_anomaly_{event_id}.parquet`** per event (columns include `event_id`, `event_label`).

**Runtime:** many Parquet reads — use a machine with **≥16 GB RAM** if possible; run after notebook 01.


In [1]:
import sys
from pathlib import Path

import pandas as pd
import yaml

_cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Repo root not found")
sys.path.insert(0, str(REPO_ROOT))

from src.io.smap_weekly_parquet import event_week_columns, load_smap_year_metadata
from src.modeling.task3_smap_anomalies import (
    baseline_climatology_iso_weeks,
    compute_event_anomalies,
    event_windows_from_cfg,
)

with open(REPO_ROOT / "configs" / "task3_soil_moisture.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

b0, b1 = int(cfg["baseline"]["period"][0]), int(cfg["baseline"]["period"][1])
events = event_windows_from_cfg(cfg)
print("Configured events:", [e["id"] for e in events])

panel_pq = REPO_ROOT / cfg["output"]["processed_dir"] / "task3_pixel_panel.parquet"
if not panel_pq.is_file():
    raise FileNotFoundError(f"Run notebook 01 first: missing {panel_pq}")
pixels = pd.read_parquet(panel_pq)

iso_union: set[int] = set()
specs_by_id: dict[str, list] = {}
for ev in events:
    y = int(ev["year"])
    meta_y = load_smap_year_metadata(REPO_ROOT, y)
    specs = event_week_columns(meta_y, str(ev["start_date"]), str(ev["end_date"]))
    specs_by_id[str(ev["id"])] = specs
    iso_union.update(w for (_, _, w) in specs)
    print(ev["id"], "year", y, "week columns", len(specs))

iso_weeks = sorted(iso_union)
print("Union ISO weeks for climatology:", iso_weeks[0], "…", iso_weeks[-1], "count", len(iso_weeks))

clim = baseline_climatology_iso_weeks(
    REPO_ROOT, range(b0, b1 + 1), iso_weeks, pixels[["iy", "ix"]]
)
out_dir = REPO_ROOT / cfg["output"]["processed_dir"]
out_dir.mkdir(parents=True, exist_ok=True)
clim_pq = out_dir / "smap_climatology.parquet"
clim.to_parquet(clim_pq, index=False)
print("Wrote", clim_pq.relative_to(REPO_ROOT), "rows", len(clim))

for ev in events:
    eid = str(ev["id"])
    y = int(ev["year"])
    specs = specs_by_id[eid]
    anom = compute_event_anomalies(REPO_ROOT, y, specs, clim, pixels)
    anom["event_id"] = eid
    anom["event_label"] = str(ev.get("label", eid))
    anom_pq = out_dir / f"smap_anomaly_{eid}.parquet"
    anom.to_parquet(anom_pq, index=False)
    print("Wrote", anom_pq.relative_to(REPO_ROOT), "rows", len(anom))


Configured events: ['midwest_flood_2019', 'plains_drought_2022']
midwest_flood_2019 year 2019 week columns 18
plains_drought_2022 year 2022 week columns 13
Union ISO weeks for climatology: 14 … 35 count 22
Wrote data\processed\task3\smap_climatology.parquet rows 45850464
Wrote data\processed\task3\smap_anomaly_midwest_flood_2019.parquet rows 37514016
Wrote data\processed\task3\smap_anomaly_plains_drought_2022.parquet rows 27093456
